<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/05_feature_engineering_v0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05_feature_engineering.ipynb — Engenharia de Atributos (5 min)

## Objetivo
Construir atributos derivados para modelagem supervisionada a partir da série temporal contínua (5 minutos) e da segmentação de episódios críticos.

## Entradas (artefatos dos Notebooks 03 e 04)
- `window_5min_series.parquet`
- `episodes_detected.parquet`

## Saídas (artefatos deste Notebook)
- `window_5min_features.parquet`
- `05_feature_engineering_summary.json`

## Compatibilidade com o pipeline existente (00 → 04)
Este notebook mantém a granularidade oficial de 5 minutos e:
- preserva `bucket_id` como eixo temporal
- utiliza `n_failed` como métrica base
- incorpora `is_critical` como variável derivada
- gera atributos rolling e temporais

## Features construídas
- Lags (1, 2, 3)
- Rolling mean (1h)
- Rolling std (1h)
- Variação percentual
- Intensidade normalizada

## Observações de reprodutibilidade
- Bootstrap via Drive + repositório `PPCOMP_DM`
- Uso de `src.paths` e `ensure_dirs()`
- Persistência em `03-features`
- Resumo estatístico salvo em `04-reports`
- Seed fixa

In [ ]:
# ============================================================
# 05_feature_engineering.ipynb
# Engenharia de atributos (5-min)
# Pipeline PPCOMP_DM (Google Cluster Trace) - V0 baseado em n_failed
# ============================================================

# -----------------------------
# 0) BOOTSTRAP (Colab + Repo)
# -----------------------------
from pathlib import Path
import os
import sys
import subprocess
import importlib
import random
import numpy as np
import pandas as pd
import json

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("[Bootstrap] Google Drive já montado.")

REPO_DIR = Path("/content/drive/MyDrive/Mestrado/PPCOMP_DM")
GITHUB_REPO = "https://github.com/sergiocostaifes/PPCOMP_DM.git"

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    print(f"[Bootstrap] Clonando repositório em: {REPO_DIR}")
    subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)
else:
    try:
        print("[Bootstrap] Atualizando repositório (git pull).")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    except Exception as e:
        print("[Bootstrap] Aviso: não foi possível atualizar via git pull:", e)

os.chdir(str(REPO_DIR))
print("[Bootstrap] CWD =", os.getcwd())

repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
importlib.invalidate_caches()

from src.paths import FEATURES_PATH, REPORTS_PATH, ensure_dirs
ensure_dirs()

print("FEATURES_PATH =", FEATURES_PATH)
print("REPORTS_PATH =", REPORTS_PATH)

def log(msg: str) -> None:
    print(f"[05_feature_engineering] {msg}")

# -----------------------------
# 1) Inputs
# -----------------------------
SERIES_FILE = FEATURES_PATH / "window_5min_series.parquet"
assert SERIES_FILE.exists(), f"Arquivo não encontrado: {SERIES_FILE}"

df = pd.read_parquet(SERIES_FILE).sort_values("bucket_id").reset_index(drop=True)
log(f"Shape série base: {df.shape}")

assert "bucket_id" in df.columns
assert "n_failed" in df.columns

# -----------------------------
# 2) Recriar criticidade (v0)
# -----------------------------
mu0 = df["n_failed"].mean()
sigma0 = df["n_failed"].std(ddof=0)
threshold0 = mu0 + 2 * sigma0

df["is_critical"] = (df["n_failed"] >= threshold0).astype("int8")
log(f"Total janelas críticas: {int(df['is_critical'].sum())}")

# -----------------------------
# 3) Construção de features
# -----------------------------
# Lags
df["lag_1"] = df["n_failed"].shift(1)
df["lag_2"] = df["n_failed"].shift(2)
df["lag_3"] = df["n_failed"].shift(3)

# Rolling 1 hora (12 janelas de 5 min)
W = 12
df["rolling_mean_1h"] = df["n_failed"].rolling(W, min_periods=1).mean()
df["rolling_std_1h"] = df["n_failed"].rolling(W, min_periods=1).std().fillna(0)

# Variação percentual
df["pct_change"] = df["n_failed"].pct_change()
df["pct_change"] = df["pct_change"].replace([np.inf, -np.inf], 0).fillna(0)

# Intensidade normalizada
mu = df["n_failed"].mean()
sigma = df["n_failed"].std(ddof=0)
df["zscore_global"] = (df["n_failed"] - mu) / (sigma if sigma > 0 else 1)

# -----------------------------
# 4) Limpeza inicial (remover NaNs de lag)
# -----------------------------
df = df.dropna().reset_index(drop=True)
log(f"Shape após feature engineering: {df.shape}")

# -----------------------------
# 5) Persistência
# -----------------------------
OUT_FILE = FEATURES_PATH / "window_5min_features.parquet"
df.to_parquet(OUT_FILE, compression="snappy", index=False)

summary = {
    "rows": int(len(df)),
    "features": list(df.columns),
    "critical_ratio": float(df["is_critical"].mean()),
    "mu": float(mu),
    "sigma": float(sigma)
}

summary_file = REPORTS_PATH / "05_feature_engineering_summary.json"
summary_file.write_text(json.dumps(summary, indent=2, ensure_ascii=False))

log("Notebook 05 finalizado com sucesso.")
df.head()

[Bootstrap] Google Drive já montado.
[Bootstrap] Atualizando repositório (git pull)...
[Bootstrap] CWD = /content/drive/MyDrive/Mestrado/PPCOMP_DM
FEATURES_PATH = /content/drive/MyDrive/Mestrado/02-datasets/03-features
REPORTS_PATH  = /content/drive/MyDrive/Mestrado/04-reports
[05_feature_engineering] Shape série base: (8918, 18)
[05_feature_engineering] Total janelas críticas: 394
[05_feature_engineering] Shape após feature engineering: (8914, 27)
[05_feature_engineering] Notebook 05 finalizado com sucesso.


,bucket_id,bucket_start_us,n_events,n_failed,n_machines,n_collections,mean_priority,mean_req_cpus,mean_req_mem,req_cpus_presence_rate,...,event_KILL_count,fail_rate,is_critical,lag_1,lag_2,lag_3,rolling_mean_1h,rolling_std_1h,pct_change,zscore_global
0,15,4500000000,44,10,41,29,249.522727,0.008850,0.003220,1.0,...,0,0.227273,0,0.323529,0.282609,0.240000,0.268353,0.043738,-0.297521,0.218397
1,16,4800000000,22,5,22,19,219.045455,0.011749,0.003593,1.0,...,1,0.227273,0,0.227273,0.323529,0.282609,0.260137,0.042099,0.000000,0.218397
2,17,5100000000,28,5,27,22,279.392857,0.010309,0.004613,1.0,...,0,0.178571,0,0.227273,0.227273,0.323529,0.246543,0.050266,-0.214286,-0.127306
3,18,5400000000,29,7,29,22,259.517241,0.007249,0.009697,1.0,...,2,0.241379,0,0.178571,0.227273,0.227273,0.245805,0.045928,0.351724,0.318531
4,19,5700000000,36,9,34,27,237.027778,0.010176,0.009284,1.0,...,0,0.250000,0,0.241379,0.178571,0.227273,0.246329,0.042547,0.035714,0.379725


## Achados do Notebook 05 — Engenharia de Atributos

Este notebook teve como objetivo enriquecer a série temporal com atributos derivados capazes de representar tendência, memória de curto prazo, variabilidade local e intensidade relativa de falhas em janelas de 5 minutos.

As features produzidas aqui formam a base observável que será utilizada pelos classificadores supervisionados nos notebooks seguintes. Em vez de utilizar apenas o valor bruto de falhas por janela, a engenharia de atributos introduz contexto temporal e histórico, o que torna a modelagem mais robusta.

### Principais grupos de atributos gerados
- **lags**: memória curta do comportamento recente
- **rolling mean**: tendência local em 1 hora
- **rolling std**: variabilidade local em 1 hora
- **pct_change**: variação relativa entre janelas consecutivas
- **atributos normalizados**: escala mais comparável entre janelas

### Interpretação
A construção dessas features permite transformar a série simples em uma representação mais rica do estado operacional do sistema. Em especial, atributos de variabilidade e tendência local são importantes para distinguir janelas normais de janelas próximas a episódios críticos.

### Limitações
- as features continuam baseadas em volume absoluto de falhas
- a janela de 1 hora é uma escolha metodológica inicial
- dependências temporais mais longas não são explicitamente modeladas

Ainda assim, este notebook estabelece uma base sólida para o treinamento supervisionado inicial.